In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

In [2]:
EMBEDDING_DIM = 128

In [4]:
num_classes = 62

### Simple CNN

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            block(3, 32), block(32, 64), block(64, 128), block(128, 256)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

In [5]:
simple_cnn_model = SimpleCNN(num_classes)

In [7]:
x = torch.randn(1, 3, 250, 250)

output = simple_cnn_model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)  # (1, num_classes)

Input shape: torch.Size([1, 3, 250, 250])
Output shape: torch.Size([1, 62])


### Resnet 18

In [8]:
def build_resnet18(num_classes: int, pretrained: bool = True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

In [9]:
resnet = build_resnet18(num_classes)

In [10]:
x = torch.randn(1, 3, 250, 250)

output = resnet(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)  # (1, num_classes)

Input shape: torch.Size([1, 3, 250, 250])
Output shape: torch.Size([1, 62])


### Embedding Net

In [11]:
class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim: int = EMBEDDING_DIM, pretrained: bool = True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        backbone.fc = nn.Linear(backbone.fc.in_features, embedding_dim)
        self.backbone = backbone

    def forward(self, x):
        return F.normalize(self.backbone(x), p=2, dim=1)

In [12]:
embedding_net = EmbeddingNet()

In [13]:
x = torch.randn(1, 3, 250, 250)

output = embedding_net(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)  # (1, num_classes)

Input shape: torch.Size([1, 3, 250, 250])
Output shape: torch.Size([1, 128])
